# "THE PRICE IS RIGHT" Dự án Capstone

Tuần này - chúng ta sẽ xây dựng một mô hình dự đoán giá của một sản phẩm từ mô tả, dựa trên dữ liệu Amazon đã được thu thập.

# Trình tự thực hiện

NGÀY 1: Thu thập dữ liệu  
NGÀY 2: Tiền xử lý dữ liệu  
NGÀY 3: Đánh giá, baseline, học máy truyền thống  
NGÀY 4: Deep Learning và LLM  
NGÀY 5: Fine-tuning một mô hình frontier  

## NGÀY 5: Fine-tuning một mô hình frontier

Hôm nay, chúng ta sẽ sử dụng API của OpenAI để fine-tune phiên bản riêng tư của GPT-4.1-nano.

### Tóm tắt quy trình của notebook

Notebook này hướng dẫn quy trình fine-tuning mô hình để ước lượng giá sản phẩm từ mô tả. Dữ liệu đầu vào được chuẩn bị ở định dạng JSONL, tải lên OpenAI, rồi khởi tạo job fine-tuning để tạo mô hình tùy chỉnh.

### Ý nghĩa chính của notebook

Mục tiêu của notebook là biến mô hình ngôn ngữ tổng quát thành một mô hình chuyên biệt cho bài toán định giá sản phẩm. Đầu ra cuối cùng là một mô hình có thể nhận mô tả sản phẩm và trả về ước tính giá phù hợp.


In [1]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

# Cell này import các thư viện cần thiết cho quá trình fine-tune và đánh giá mô hình.
# `load_dotenv` đọc biến môi trường từ file .env, `login` xác thực với Hugging Face,
# `OpenAI` tạo client để gọi API OpenAI, và `Item`, `evaluate` là các module của dự án để xử lý dữ liệu và đánh giá kết quả.


In [2]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

# Cell này thiết lập môi trường chạy dự án.
# `LITE_MODE = False` nghĩa là dùng toàn bộ dataset, không rút gọn.
# `load_dotenv()` nạp biến môi trường từ file .env, rồi `HF_TOKEN` được dùng để đăng nhập Hugging Face.
# Việc đăng nhập này cần thiết để tải dữ liệu từ Hub của Hugging Face về máy local.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# Cell này tải bộ dữ liệu từ Hugging Face về và chia thành tập train, validation, test.
# `Item.from_hub(dataset)` trả về 3 tập dữ liệu tương ứng với việc huấn luyện, kiểm định và đánh giá mô hình.
# Kết quả print giúp xác nhận số lượng mẫu của từng tập trước khi bắt đầu fine-tune.


Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [4]:
openai = OpenAI()

# Cell này khởi tạo client OpenAI để có thể gọi API fine-tuning và inference sau này.
# Từ đây, mọi thao tác với OpenAI như upload file, tạo job, hoặc gọi mô hình đều đi qua đối tượng `openai`.


# Kích thước dữ liệu

OpenAI khuyến nghị fine-tuning với một tập dữ liệu nhỏ khoảng 50-100 ví dụ.

Tôi sẽ chọn 20.000 điểm dữ liệu.

Chi phí này là $3.42 - bạn nên giữ khoảng 100 ví dụ để chi phí rất thấp!

### Tóm tắt quy trình của notebook

Ở phần này, chúng ta cân nhắc kích thước bộ dữ liệu cho việc fine-tuning. Mặc dù OpenAI gợi ý số lượng nhỏ, nhưng vì dữ liệu của bài toán rất ngắn và dễ học, ta có thể dùng một mẫu vừa đủ để tiết kiệm chi phí nhưng vẫn có hiệu quả.

### Ý nghĩa chính của notebook

Cell này giúp người học hiểu nguyên tắc: dữ liệu huấn luyện không cần quá lớn nếu ví dụ đã đủ chất lượng và bài toán không quá phức tạp.


In [6]:
# OpenAI khuyến nghị fine-tuning với khoảng 50-100 ví dụ
# Nhưng vì các ví dụ rất ngắn, tôi đề xuất dùng 100 ví dụ (và 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

# Cell này lấy một mẫu nhỏ nhưng đủ để fine-tune.
# `train[:100]` dùng 100 mẫu cho tập huấn luyện, còn `val[:50]` dùng 50 mẫu cho tập xác thực.
# Việc giảm kích thước tập dữ liệu giúp tiết kiệm chi phí và vẫn cho phép kiểm tra khả năng học của mô hình nhanh chóng.


In [7]:
len(fine_tune_train)

# Cell này kiểm tra số lượng mẫu đã lấy cho tập huấn luyện.
# Kết quả mong muốn là 100, cho thấy ta đã chọn đúng kích thước dữ liệu theo khuyến nghị của OpenAI.


100

# Bước 1

Chuẩn bị dữ liệu của chúng ta để fine-tuning ở định dạng JSONL (JSON Lines) và upload lên OpenAI.

### Tóm tắt quy trình của notebook

Ở bước này, dữ liệu sản phẩm được chuyển thành dạng hội thoại chuẩn mà mô hình có thể học được: user hỏi giá, assistant trả về giá tương ứng.

### Ý nghĩa chính của notebook

Dữ liệu fine-tune phải có định dạng đúng để mô hình hiểu rõ mục tiêu: nhận mô tả sản phẩm rồi đưa ra giá ước tính.


In [8]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

# Hàm này biến mỗi sản phẩm thành một mẫu hội thoại gồm hai phần: user đặt câu hỏi và assistant trả lời giá.
# `item.summary` là mô tả sản phẩm, còn `item.price` là nhãn giá cần mô hình học.
# Cách này giúp mô hình học trực tiếp dạng "đọc mô tả -> đưa ra con số giá".


In [9]:
messages_for(fine_tune_train[0])

# Cell này kiểm tra mẫu hội thoại đầu tiên xem dữ liệu đã đúng format chưa.
# Nếu kết quả trả về có `role` là `user` và `assistant`, nghĩa là ta đã chuẩn hóa dữ liệu phù hợp để fine-tune.


[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [10]:
# Chuyển các item thành danh sách các đối tượng json - chuỗi "jsonl"
# Mỗi dòng đại diện cho một tin nhắn theo cấu trúc:
# {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

# Hàm này lặp qua từng sản phẩm, gọi `messages_for()` để tạo dạng hội thoại, rồi nén JSON vào từng dòng.
# Mỗi dòng sau đó sẽ là một mẫu huấn luyện riêng, dạng mà OpenAI yêu cầu cho fine-tuning.


In [11]:
print(make_jsonl(train[:3]))

# Cell này in ra 3 mẫu dữ liệu đầu tiên để kiểm tra định dạng JSONL.
# Nếu chuỗi hiển thị đúng cấu trúc `{"messages": [...]}` thì dữ liệu đã sẵn sàng để upload lên OpenAI.


{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [12]:
# Chuyển các item sang jsonl và ghi vào file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

# Hàm này tạo file `.jsonl` chứa toàn bộ dữ liệu fine-tune ở định dạng mà API OpenAI chấp nhận.
# Việc lưu ra file là cần thiết vì OpenAI upload file theo cách này trước khi bắt đầu job fine-tuning.


In [13]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

# Cell này ghi tập train ra file JSONL để chuẩn bị upload lên OpenAI.
# File này sẽ chứa các ví dụ mà mô hình sẽ học từ đó.


In [14]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

# Cell này cũng ghi tập validation ra file riêng để theo dõi độ tốt của mô hình trong quá trình fine-tuning.
# Tập validation giúp đánh giá xem mô hình có học đúng hay không ngoài dữ liệu train.


In [15]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

# Cell này upload file dữ liệu train lên OpenAI để chuẩn bị cho job fine-tuning.
# Sau khi upload, OpenAI trả về một object chứa ID của file, và ID này sẽ được dùng trong lệnh tạo job.


In [16]:
train_file

# Cell này hiển thị thông tin file train đã upload.
# Chúng ta cần xác nhận file đã sẵn sàng để dùng trong quá trình fine-tune.


FileObject(id='file-Crm3yFWYXtxPuuqNxW1mGH', bytes=55219, created_at=1789889761, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [17]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

# Cell này upload file validation lên OpenAI để dùng khi đánh giá mô hình trong quá trình training.
# Tập validation giúp kiểm tra xem mô hình không chỉ học thuộc dữ liệu train mà còn tổng quát tốt hơn.


In [18]:
validation_file

# Cell này hiển thị file validation đã được upload.
# Đây là dấu hiệu cho thấy quá trình chuẩn bị dữ liệu đã hoàn tất và sẵn sàng để tạo job fine-tuning.


FileObject(id='file-3AgtmSoYLW8p8vELbqb8Ef', bytes=27686, created_at=1789889795, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Đây là đường dẫn để xem các file đã upload lên OpenAI trong dashboard.
# Nếu cần kiểm tra dữ liệu đã được lưu đúng hay chưa, bạn có thể mở đường link này trong trình duyệt.


# Bước 2

## Bây giờ là lúc Fine-tune!

### Tóm tắt quy trình của notebook

Sau khi dữ liệu đã sẵn sàng, ta bắt đầu tạo job fine-tuning trên OpenAI. Job này sẽ dùng tập train và validation để điều chỉnh mô hình gốc theo mục tiêu định giá sản phẩm.

### Ý nghĩa chính của notebook

Đây là bước cốt lõi của notebook: biến mô hình nền thành mô hình thích nghi với bài toán thực tế của dự án.


In [21]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

# Cell này tạo một job fine-tuning trên OpenAI nếu API vẫn cho phép.
# `training_file` và `validation_file` là ID của các file JSONL đã upload trước đó.
# `model="gpt-4.1-nano-2025-04-14"` chọn mô hình nền để tinh chỉnh.
# `seed=42` đảm bảo tính lặp lại của quá trình training, còn `n_epochs=1` và `batch_size=1` giữ cho việc huấn luyện nhẹ và tiết kiệm chi phí.
# `suffix="pricer"` đặt tên cho phiên bản mô hình mới tạo ra.
# Nếu API trả về 403, đó không phải lỗi code trong notebook mà là vì OpenAI đã ngừng dịch vụ fine-tuning tại tổ chức này.


PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

In [ ]:
openai.fine_tuning.jobs.list(limit=1)

# Cell này liệt kê job fine-tuning gần nhất.
# Đây là cách đơn giản để xác nhận job đã được tạo thành công hoặc đang chạy.


In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

# Cell này lấy ID của job mới nhất để theo dõi trạng thái và tiến độ fine-tuning.
# `job_id` sẽ được sử dụng ở các cell sau để truy vấn trạng thái training hoặc kết quả cuối cùng.


In [ ]:
job_id

# Cell này hiển thị `job_id`, giúp ta dễ theo dõi job trên dashboard hoặc qua API.
# Nếu job đang chạy hoặc đã hoàn thành, ta có thể dùng ID này để truy vấn chi tiết.


In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

# Cell này truy vấn trạng thái chi tiết của job fine-tuning.
# Thông tin trả về cho biết job đang ở trạng thái nào: queued, running, succeeded, failed,...


In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

# Cell này lấy các sự kiện log của job trong 10 bản ghi gần nhất.
# Đây là cách tốt để xem quá trình training diễn ra như thế nào, từ bắt đầu đến khi hoàn tất.


https://platform.openai.com/finetune

# Đây là trang dashboard của OpenAI nơi ta có thể theo dõi job fine-tuning theo giao diện.
# Nếu bạn muốn xem tiến độ và kết quả ngay trên web, hãy mở đường link này trong browser.


# Bước 3

Kiểm tra mô hình đã fine-tune.

### Tóm tắt quy trình của notebook

Sau khi job hoàn thành, ta lấy tên mô hình đã fine-tune và thử đưa dữ liệu test vào để xem mô hình trả về giá như thế nào.

### Ý nghĩa chính của notebook

Đây là bước đánh giá thực tế: kiểm tra xem mô hình mới có giải quyết đúng bài toán định giá sản phẩm hay không.


In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

# Cell này lấy tên của mô hình đã được fine-tune.
# Sau khi job thành công, OpenAI sẽ tạo ra một model mới và gán tên cho nó để ta có thể gọi trực tiếp.


In [ ]:
fine_tuned_model_name

# Cell này hiển thị tên mô hình fine-tuned vừa lấy được.
# Đây là đối tượng mô hình mà ta sẽ dùng cho phần inference ở các cell tiếp theo.


In [ ]:
# Câu prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

# Hàm này tạo câu hỏi cho mô hình khi chạy inference trên một sản phẩm mới.
# Mô hình sẽ nhận mô tả sản phẩm và chỉ cần trả về giá, không cần giải thích.
# Đây là định dạng tương tự như khi ta huấn luyện, giúp mô hình phản hồi theo đúng kỳ vọng.


In [ ]:
# Thử chạy xem sao

test_messages_for(test[0])

# Cell này kiểm tra prompt được tạo cho một ví dụ test đầu tiên.
# Nếu output trông hợp lý, ta có thể đưa prompt này vào mô hình fine-tuned và xem câu trả lời thực tế.


In [ ]:
# Hàm suy luận


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

# Hàm này gọi mô hình fine-tuned với câu hỏi về giá sản phẩm.
# `max_tokens=7` là đủ để mô hình trả về một con số ngắn, ví dụ `$24.99` hoặc `29.99`.
# Kết quả được trả về dưới dạng chuỗi text, sau đó có thể so sánh với giá thật để đánh giá độ chính xác.


In [ ]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

# Cell này so sánh giá thật với giá mà mô hình dự đoán cho cùng một sản phẩm.
# Đây là cách trực quan nhất để kiểm tra xem mô hình có đang đưa ra con số gần với giá thực tế hay không.


In [ ]:
evaluate(gpt_4__1_nano_fine_tuned, test)

# Cell này đánh giá hiệu suất của mô hình trên tập test.
# Hàm `evaluate` so sánh kết quả dự đoán với giá thật để tính toán mức độ chính xác của mô hình.


In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000

# Đây là các số liệu hiệu suất đã ghi lại từ các lần thử khác nhau.
# Chúng cho thấy mô hình fine-tuned có thể đạt độ chính xác khá tốt trên tập test, và thêm dữ liệu/kiểu mô hình phù hợp có thể cải thiện kết quả hơn nữa.
